# Lazy Portfolio Panel — Viewer
Analisi comparativa dei risultati prodotti da `iq lazy-analyze`.
**Non calcola nulla** — legge i file gia' prodotti dalla CLI:
- `outputs/lazy_analysis/*/classification_*.csv`

In [ ]:
%run _bootstrap_dev.ipynb

## §1 — Configurazione

In [ ]:
from pathlib import Path
from datetime import datetime
from IPython.display import display
import plotly.express as px
import glob

ROOT        = Path().resolve().parent.parent   # investia-quant/
LAZY_DIR    = ROOT / 'outputs' / 'lazy_analysis'
EXPORT_DIR  = ROOT / 'outputs' / 'lazy_panel_exports'
EXPORT_DIR.mkdir(parents=True, exist_ok=True)

CSV_PATTERN     = '*/classification_*.csv'
FILTER_PROMOTED = 'ALL'       # 'ALL' | 'PROMOTED' | 'FAILED'
SORT_BY         = 'Sharpe'    # 'Sharpe' | 'CAGR%' | 'MaxDD%' | 'DSR'
SORT_ASC        = False

print(f'Lazy dir : {LAZY_DIR}')
print(f'Export   : {EXPORT_DIR}')
print(f'Filtro   : {FILTER_PROMOTED} | Sort: {SORT_BY} asc={SORT_ASC}')

## §2 — Classifica

In [ ]:
df = load_lazy_classifications(LAZY_DIR, CSV_PATTERN, FILTER_PROMOTED, SORT_BY, SORT_ASC)
print(f'Totale    : {len(df)}')
print(f'Promossi  : {(df["Verdetto"]=="PROMOSSO").sum()}')
print(f'Rigettati : {(df["Verdetto"]=="RIGETTATO").sum()}')
my_display(df)

## §3 — Scatter Sharpe vs CAGR%

In [ ]:
fig = px.scatter(
    df, x='Sharpe', y='CAGR%',
    color='Verdetto',
    color_discrete_map={'PROMOSSO': '#28a745', 'RIGETTATO': '#dc3545'},
    hover_name='Nome',
    hover_data=['MaxDD%', 'PLoss5y%', 'MinSafeHorizon', 'DSR', 'BestFreq'],
    size='MaxDD%',
    title='Lazy Portfolio — Sharpe vs CAGR%'
)
fig.update_layout(height=500)
fig.show()

## §4 — Equity curve PTF promossi

In [ ]:
TOP_N = 14
plot_lazy_equity_curves(
    promoted_df=df,
    registry=L_PORTFOLIO_REGISTRY,
    top=TOP_N,
    sort_by='Sharpe',
    mode='overlay',
    common_period_only=True
)

## §5 — Export promossi

In [ ]:
promoted = df[df['Verdetto'] == 'PROMOSSO'].copy()
ts = datetime.now().strftime('%Y%m%d_%H%M%S')

csv_path = EXPORT_DIR / f'promoted_{ts}.csv'
promoted.drop(columns=['_run', '_file'], errors='ignore').to_csv(csv_path, index=False)
print(f'CSV salvato: {csv_path}')
print()

display(style_lazy_classification(
    promoted.drop(columns=['_run', '_file'], errors='ignore')
))

## §7 — Proiezione capitale futuro

In [ ]:
CAPITALE_INIZIALE = 160_000.0
ORIZZONTE_ANNI = 30
CACHE_DIR = ROOT / 'outputs' / 'lazy_cache'

ptf_da_confrontare = [
    'lazy_conservative_40_30_30',
    'lazy_balanced_60_20_20',
    'lazy_aggressive_80_10_10',
    'lazy_full_equity_95_5',
]

projection = project_lazy_capital(
    ptf_names=ptf_da_confrontare,
    cache_dir=CACHE_DIR,
    initial_capital=CAPITALE_INIZIALE,
    horizon_years=ORIZZONTE_ANNI,
    percentiles=(10, 50, 70),
    plot=True,
)

# projection['fig'].update_layout(margin=dict(b=80))  # se la funzione restituisce la fig